# FoodOps AI — Análise Logística com SQL

**Tecnologias:** Python, Pandas e SQLite | **Ambiente:** Google Colab

## Objetivo

Este notebook complementa a análise de Machine Learning do FoodOps AI por meio de consultas SQL sobre os pedidos históricos de e-commerce. O objetivo é responder a perguntas de negócio sobre atrasos, distribuição geográfica, evolução temporal e priorização de pedidos.

**Fonte dos dados:** `foodops_ai_dashboard.csv`, gerado na etapa anterior do projeto. O arquivo contém pedidos do conjunto de teste e a classificação de prioridade produzida pelo modelo. A análise se restringe aos pedidos históricos entregues presentes nesse arquivo.

**Roteiro:** importar o CSV, criar a tabela `pedidos` no SQLite, validar os registros e responder às perguntas de negócio usando SQL. As consultas são executadas por `pd.read_sql_query()`; o processamento dos agrupamentos e indicadores ocorre no banco SQLite.


## 1. Preparação — Importação do arquivo

**Objetivo:** disponibilizar o arquivo `foodops_ai_dashboard.csv` no ambiente do Colab. A célula solicita o upload do CSV; selecione o arquivo exportado na etapa anterior do FoodOps AI.


In [ ]:
import pandas as pd
import sqlite3
from google.colab import files

uploaded = files.upload()

Saving foodops_ai_dashboard.csv to foodops_ai_dashboard (1).csv


**Resultado da preparação:** o arquivo foi disponibilizado para as consultas. Em novas sessões do Colab, será necessário carregar o CSV novamente, pois os arquivos locais do ambiente podem ser apagados.


## 2. Preparação — Criação da tabela e conferência dos registros

**Pergunta de validação:** quantos pedidos foram importados para o banco de dados?

O CSV é carregado com Pandas e copiado para uma tabela SQLite chamada `pedidos`. A consulta `SELECT COUNT(*)` verifica a quantidade de registros disponíveis para as análises seguintes.

**Resultado:** 19.294 registros, conforme a execução original do notebook. Esse total é a base de comparação para os indicadores calculados a seguir.


In [ ]:
# Carregar o dataset
df = pd.read_csv("foodops_ai_dashboard.csv")

# Criar conexão com o banco SQLite
conn = sqlite3.connect("foodops_ai.db")

# Criar tabela SQL
df.to_sql("pedidos", conn, if_exists="replace", index=False)

# Verificar quantidade de registros
resultado = pd.read_sql_query("""
    SELECT COUNT(*) AS total_pedidos
    FROM pedidos
""", conn)

print(resultado)

   total_pedidos
0          19294


**Interpretação:** a tabela `pedidos` contém **19.294 registros**, em concordância com o conjunto utilizado na análise do modelo. A execução desta etapa prepara a base para todas as consultas seguintes.


## 3. Pergunta 1 — Quantos pedidos atrasaram e qual foi a taxa geral de atraso?

**Objetivo:** medir a frequência de atrasos no conjunto analisado.

A consulta utiliza `COUNT(*)` para contar os pedidos, `SUM(atraso_real)` para contar os atrasados (variável binária 0/1) e `ROUND()` para apresentar a taxa em porcentagem com duas casas decimais.


In [ ]:
consulta = """
SELECT
    COUNT(*) AS total_pedidos,
    SUM(atraso_real) AS pedidos_atrasados,
    ROUND(
        100.0 * SUM(atraso_real) / COUNT(*),
        2
    ) AS taxa_atraso_percentual
FROM pedidos;
"""

resultado = pd.read_sql_query(consulta, conn)
display(resultado)

,total_pedidos,pedidos_atrasados,taxa_atraso_percentual
0,19294,674,3.49


### Resposta à pergunta 1

Foram analisados **19.294 pedidos**, dos quais **674 atrasaram**, resultando em uma **taxa geral de atraso de 3,49%**.

**Conclusão de negócio:** essa taxa funciona como referência para comparar estados, meses e grupos de prioridade. Ela descreve apenas os pedidos presentes no conjunto de teste, não necessariamente toda a operação de e-commerce.


## 4. Pergunta 2 — Quais estados apresentaram as maiores taxas de atraso?

**Objetivo:** comparar a incidência de atrasos entre as unidades federativas dos clientes.

A consulta agrupa os registros com `GROUP BY customer_state`, calcula o volume e a taxa de atrasos em cada estado e ordena os resultados pela maior taxa (`ORDER BY ... DESC`).


In [ ]:
consulta_estados = """
SELECT
    customer_state AS estado,
    COUNT(*) AS total_pedidos,
    SUM(atraso_real) AS pedidos_atrasados,
    ROUND(
        100.0 * SUM(atraso_real) / COUNT(*),
        2
    ) AS taxa_atraso_percentual
FROM pedidos
GROUP BY customer_state
ORDER BY taxa_atraso_percentual DESC;
"""

resultado_estados = pd.read_sql_query(consulta_estados, conn)
display(resultado_estados)

,estado,total_pedidos,pedidos_atrasados,taxa_atraso_percentual
0,PI,84,11,13.10
1,MA,115,14,12.17
2,MS,137,13,9.49
3,AL,66,4,6.06
4,TO,55,3,5.45
5,SP,8921,405,4.54
6,PB,113,5,4.42
7,BA,638,28,4.39
8,PE,322,14,4.35
9,ES,360,14,3.89


### Resposta à pergunta 2

Os maiores percentuais observados foram **PI: 13,10% (11 de 84 pedidos)**, **MA: 12,17% (14 de 115)** e **MS: 9,49% (13 de 137)**. Já **SP apresentou 405 atrasos em 8.921 pedidos (4,54%)**, combinando grande volume de ocorrências com uma taxa inferior à dos três estados citados.

**Conclusão de negócio:** é importante analisar **taxa e volume simultaneamente**. Estados com poucos pedidos podem apresentar taxas altas mesmo com poucos atrasos; portanto, esses resultados não demonstram, por si só, uma diferença estrutural de desempenho logístico.


## 5. Pergunta 3 — Os pedidos de alta prioridade concentraram mais atrasos?

**Objetivo:** verificar se a classificação de prioridade do modelo separa grupos com diferentes taxas de atraso.

A consulta usa `GROUP BY prioridade` para comparar quantidade de pedidos, atrasos observados e taxa de atraso nos grupos **Alta** e **Normal**.


In [ ]:
consulta_prioridade = """
SELECT
    prioridade,
    COUNT(*) AS total_pedidos,
    SUM(atraso_real) AS pedidos_atrasados,
    ROUND(
        100.0 * SUM(atraso_real) / COUNT(*),
        2
    ) AS taxa_atraso_percentual
FROM pedidos
GROUP BY prioridade
ORDER BY taxa_atraso_percentual DESC;
"""

resultado_prioridade = pd.read_sql_query(
    consulta_prioridade, conn
)

display(resultado_prioridade)

,prioridade,total_pedidos,pedidos_atrasados,taxa_atraso_percentual
0,Alta,1930,138,7.15
1,Normal,17364,536,3.09


### Resposta à pergunta 3

O grupo de **alta prioridade** reuniu **1.930 pedidos**, com **138 atrasos (7,15%)**. O grupo **normal** reuniu **17.364 pedidos**, com **536 atrasos (3,09%)**.

**Conclusão de negócio:** o grupo prioritário apresentou taxa de atraso aproximadamente **2,31 vezes** a do grupo normal. Isso sugere que o ranking do modelo ajuda a concentrar pedidos com maior incidência de atrasos, embora muitos atrasos permaneçam fora do grupo prioritário.


## 6. Pergunta 4 — Como a taxa de atraso evoluiu ao longo dos meses?

**Objetivo:** identificar mudanças temporais na frequência de atrasos.

`SUBSTR(order_purchase_timestamp, 1, 7)` extrai ano e mês do texto da data de compra. A consulta agrupa por mês, calcula volumes e taxas e apresenta os meses em ordem cronológica.


In [ ]:
consulta_mensal = """
SELECT
    SUBSTR(order_purchase_timestamp, 1, 7) AS mes,
    COUNT(*) AS total_pedidos,
    SUM(atraso_real) AS pedidos_atrasados,
    ROUND(
        100.0 * SUM(atraso_real) / COUNT(*),
        2
    ) AS taxa_atraso_percentual
FROM pedidos
GROUP BY SUBSTR(order_purchase_timestamp, 1, 7)
ORDER BY mes;
"""

resultado_mensal = pd.read_sql_query(consulta_mensal, conn)
display(resultado_mensal)

,mes,total_pedidos,pedidos_atrasados,taxa_atraso_percentual
0,2018-05,691,2,0.29
1,2018-06,6096,71,1.16
2,2018-07,6156,208,3.38
3,2018-08,6351,393,6.19


### Resposta à pergunta 4

| Mês | Pedidos | Atrasados | Taxa de atraso |
|---|---:|---:|---:|
| 2018-05 | 691 | 2 | 0,29% |
| 2018-06 | 6.096 | 71 | 1,16% |
| 2018-07 | 6.156 | 208 | 3,38% |
| 2018-08 | 6.351 | 393 | 6,19% |

**Conclusão de negócio:** agosto apresentou tanto o **maior número de atrasos (393)** quanto a **maior taxa (6,19%)** entre os meses avaliados. A tendência crescente merece investigação, mas esses dados não permitem estabelecer sua causa nem generalizá-la para todos os pedidos da operação.


## 7. Pergunta 5 — Qual foi a cobertura dos atrasos no grupo prioritário e qual o ganho em relação à média?

**Objetivo:** reproduzir em SQL os indicadores de priorização do modelo.

- **Recall@10%:** proporção dos atrasos reais que está entre os pedidos de alta prioridade.
- **Lift@10%:** taxa de atraso do grupo de alta prioridade dividida pela taxa geral de atraso.

A consulta utiliza `CASE WHEN` para selecionar o grupo prioritário e `NULLIF()` para evitar divisão por zero.


In [ ]:
consulta_metricas = """
SELECT
    ROUND(
        100.0 * SUM(
            CASE WHEN prioridade = 'Alta'
            THEN atraso_real ELSE 0 END
        ) / NULLIF(SUM(atraso_real), 0),
        2
    ) AS recall_top10_percentual,

    ROUND(
        (
            1.0 * SUM(
                CASE WHEN prioridade = 'Alta'
                THEN atraso_real ELSE 0 END
            ) / NULLIF(SUM(
                CASE WHEN prioridade = 'Alta'
                THEN 1 ELSE 0 END
            ), 0)
        ) /
        (1.0 * SUM(atraso_real) / NULLIF(COUNT(*), 0)),
        2
    ) AS lift_top10
FROM pedidos;
"""

resultado_metricas = pd.read_sql_query(consulta_metricas, conn)
display(resultado_metricas)

,recall_top10_percentual,lift_top10
0,20.47,2.05


### Resposta à pergunta 5

O SQL retornou **Recall@10% = 20,47%** e **Lift@10% = 2,05x**, reproduzindo os indicadores obtidos na avaliação do modelo.

- **Recall@10% de 20,47%:** os 1.930 pedidos prioritários continham 138 dos 674 atrasos observados.
- **Lift@10% de 2,05x:** a taxa de atraso dos pedidos prioritários (7,15%) foi cerca de duas vezes a taxa geral (3,49%).

**Conclusão de negócio:** o modelo mostrou capacidade de concentrar atrasos em uma fila de monitoramento menor, mas **79,53% dos atrasos** ficaram fora desse grupo. O ranking é uma ferramenta experimental de priorização, não uma garantia de identificação de todos os atrasos.


## Conclusão geral

As consultas SQL confirmaram a consistência dos principais indicadores do FoodOps AI com os resultados de Machine Learning e do dashboard Power BI. O trabalho demonstrou o uso de **agregações, agrupamentos, ordenação, extração de períodos, expressões condicionais e cálculos percentuais em SQL**.

**Síntese:** foram analisados **19.294 pedidos**, com **674 atrasos (3,49%)**. O grupo de alta prioridade concentrou **138 atrasos em 1.930 pedidos**, atingindo **Recall@10% de 20,47%** e **Lift@10% de 2,05x**.

**Limitações:** os dados correspondem a um conjunto histórico de pedidos entregues; as taxas variam entre períodos e grupos; o ranking de risco não deve ser interpretado como probabilidade absoluta calibrada. Os resultados não representam validação de uma operação em produção.

**Reprodução:** execute as células na ordem, disponibilizando antes o arquivo `foodops_ai_dashboard.csv`. O banco SQLite `foodops_ai.db` é criado localmente durante a execução.
